# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EgeGln365/FlyRank_AI_Internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

For my Refresh / Content Opportunity Scoring lane, one row in the modeling
frame represents one content item at a defined decision point.

I will use March 2026 as my development month. The warehouse contains
daily performance observations, so I will summarize the historical
observations into content-level features for each content item.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### Features

For the first version of this lane, I will use Google Search Console (GSC) metrics because they are consistently available and directly describe search performance.

Planned feature fields:
- `gsc_impressions`
- `gsc_clicks`
- `gsc_avg_position`

### Label

The target is a proxy label named `is_declining`, which will be created during feature engineering rather than taken directly from the warehouse.

The following intermediate columns will be created:

- `imp_prev30`: Total Google Search Console impressions during the previous 30-day period.
- `imp_last30`: Total Google Search Console impressions during the most recent 30-day period.

The proxy label will then be defined as:

- `is_declining = 1` if `imp_last30 < 0.8 × imp_prev30`
- `is_declining = 0` otherwise.

This proxy is used because the warehouse does not contain a predefined target column for declining content.

### Context

The following fields provide context for each record and are not used as predictive features:

- `client_hash_id`: Identifies the client.
- `content_hash_id`: Identifies the content item.
- `report_date`: Defines the observation date.
- `month`: Used to select the development period (March 2026).

### Excluded

The following fields are excluded from the first version of the model:

- All GA4-related fields because their values are unavailable or incomplete for many records in the current warehouse release.
- Identifier fields (`client_hash_id` and `content_hash_id`) are excluded as predictive features because they only identify records.
- The label (`is_declining`) and any variables derived from it are excluded from the feature set to prevent target leakage.

In [2]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)

HF_TOKEN = os.getenv("HF_TOKEN")


if HF_TOKEN is None:
    raise ValueError("HF_TOKEN bulunamadı. .env dosyanı kontrol et.")


In [3]:
import duckdb

con = duckdb.connect()

con.execute(
    f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

In [4]:
REL = "hf://datasets/FlyRank/internship-warehouse"

TABLES = {
    "dim_clients": f"read_parquet('{REL}/dim_clients.parquet')",
    "dim_content": f"read_parquet('{REL}/dim_content.parquet')",
    "fact_daily": f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    "fact_daily_sample": f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    "fact_query_90d": f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

In [5]:
daily_schema = con.sql(f""" 
    DESCRIBE
    SELECT *
    FROM {TABLES["fact_daily"]}

""").df()

daily_schema

,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


In [6]:
con.sql(f"""
SELECT *
FROM {TABLES["fact_daily"]}
WHERE month = '2026-03'
LIMIT 5
""").df()

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,gsc_avg_position,ga4_pageviews,ga4_sessions,ga4_users,ga4_engaged_sessions,ga4_total_engagement_sec,sessions_organic,sessions_direct,sessions_referral,sessions_social,sessions_paid,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,<NA>,20,0,67,3.350000,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,<NA>,1,0,0,0.000000,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,False,True,<NA>,125,1,616,4.928000,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,True,False,True,<NA>,7,0,28,4.000000,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,True,False,True,<NA>,11,0,25,2.272727,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03


In [7]:
con.sql(f"""
SELECT *
FROM {TABLES["fact_daily"]}
WHERE month='2026-03'
AND ga4_data_available IS TRUE
LIMIT 5
""").df()

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,gsc_avg_position,ga4_pageviews,ga4_sessions,ga4_users,ga4_engaged_sessions,ga4_total_engagement_sec,sessions_organic,sessions_direct,sessions_referral,sessions_social,sessions_paid,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-03-01,client_65de48885f4ef01b,content_09be8cc7fcb222af,True,True,False,True,0,0,0,NaN,1,1,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,2026-03
1,2026-03-01,client_65de48885f4ef01b,content_851afac9fe13612e,True,True,False,True,0,0,0,NaN,1,1,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,2026-03
2,2026-03-01,client_65de48885f4ef01b,content_cee6c6fc8c51af14,True,True,False,True,0,0,0,NaN,1,1,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,2026-03
3,2026-03-01,client_65de48885f4ef01b,content_5e120e972f11f833,True,True,False,True,0,0,0,NaN,1,1,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,2026-03
4,2026-03-01,client_65de48885f4ef01b,content_16a7291bb6ecaebe,True,True,False,True,0,0,0,NaN,1,1,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,2026-03


In [8]:
content_schema = con.sql(f"""
    DESCRIBE
    SELECT *
    FROM {TABLES["dim_content"]}
""").df()

content_schema

,column_name,column_type,null,key,default,extra
0,client_hash_id,VARCHAR,YES,None,None,None
1,content_hash_id,VARCHAR,YES,None,None,None
2,keyword_hash_id,VARCHAR,YES,None,None,None
3,url_hash_id,VARCHAR,YES,None,None,None
4,keyword_char_count,BIGINT,YES,None,None,None
5,keyword_token_count,BIGINT,YES,None,None,None
6,url_char_count,BIGINT,YES,None,None,None
7,content_created_date,DATE,YES,None,None,None
8,content_updated_date,DATE,YES,None,None,None
9,content_type,VARCHAR,YES,None,None,None


In [9]:
d_schema = con.sql(f"""
    DESCRIBE
    SELECT *
    FROM {TABLES["fact_query_90d"]}
""").df()

d_schema

,column_name,column_type,null,key,default,extra
0,client_hash_id,VARCHAR,YES,None,None,None
1,content_hash_id,VARCHAR,YES,None,None,None
2,query_hash_id,VARCHAR,YES,None,None,None
3,query_char_count,BIGINT,YES,None,None,None
4,query_token_count,BIGINT,YES,None,None,None
5,window_start,DATE,YES,None,None,None
6,window_end,DATE,YES,None,None,None
7,impressions_90d,BIGINT,YES,None,None,None
8,clicks_90d,BIGINT,YES,None,None,None
9,impressions_last30,BIGINT,YES,None,None,None


In [10]:
con.sql(f"""
    SELECT DISTINCT window_start, window_end
    FROM {TABLES["fact_query_90d"]}
    ORDER BY window_end
    LIMIT 30
""").df()

,window_start,window_end
0,2026-04-02,2026-06-30


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [11]:
grain_check = con.sql(f"""
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        COUNT(*) AS row_count
    FROM {TABLES["fact_daily"]}
    WHERE month = '2026-03'
    GROUP BY
        report_date,
        client_hash_id,
        content_hash_id
    HAVING COUNT(*) > 1
""").df()

print(f"Duplicate grain combinations: {len(grain_check):,}")
grain_check.head()


Duplicate grain combinations: 0


,report_date,client_hash_id,content_hash_id,row_count


In [12]:
slice_summary = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT client_hash_id) AS unique_clients,
        COUNT(DISTINCT content_hash_id) AS unique_contents,
        MIN(report_date) AS start_date,
        MAX(report_date) AS end_date
    FROM {TABLES["fact_daily"]}
    WHERE month = '2026-03'
""").df()

slice_summary

,total_rows,unique_clients,unique_contents,start_date,end_date
0,9841378,55,331437,2026-03-01,2026-03-31


In [13]:
availability_check = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (
            WHERE gsc_data_available IS TRUE
        ) AS gsc_available_rows,
        COUNT(*) FILTER (
            WHERE gsc_data_available IS NOT TRUE
        ) AS gsc_unavailable_rows,
        ROUND(
            100.0 * COUNT(*) FILTER (
                WHERE gsc_data_available IS TRUE
            ) / COUNT(*),
            2
        ) AS gsc_available_pct
    FROM {TABLES["fact_daily"]}
    WHERE month = '2026-03'
""").df()

availability_check

,total_rows,gsc_available_rows,gsc_unavailable_rows,gsc_available_pct
0,9841378,3611061,6230317,36.69


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

### Data limits

This data has several important limitations.

First, GSC data is available for only 36.69% of the March 2026 daily rows. Therefore, the final feature frame represents only the subset of clients and content items with available GSC history and may not represent the full warehouse population.

Second, clients and content items do not necessarily have equal amounts of historical data. Some records may have a longer and more complete history than others, so comparisons may be affected by unbalanced observation windows.

Third, an observed decline in impressions does not prove that refreshing the content will improve its performance. The proxy label supports prioritization and decision-making, but it does not establish a causal effect.

Finally, the feature window and outcome window must remain separate. If information from the outcome period is included as a feature, the model would suffer from target leakage and produce unrealistically high results.

In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

In [15]:
print("Defined warehouse sources:")

for name, source in TABLES.items():
    print(f"- {name}: {source}")

Defined warehouse sources:
- dim_clients: read_parquet('hf://datasets/FlyRank/internship-warehouse/dim_clients.parquet')
- dim_content: read_parquet('hf://datasets/FlyRank/internship-warehouse/dim_content.parquet')
- fact_daily: read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet')
- fact_daily_sample: read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance_sample.parquet')
- fact_query_90d: read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_query_90d.parquet')


In [16]:
table_counts = []

for name, source in TABLES.items():
    row_count = con.sql(f"""
        SELECT COUNT(*)
        FROM {source}
    """).fetchone()[0]

    table_counts.append({
        "table_name": name,
        "row_count": row_count,
    })

pd.DataFrame(table_counts)

,table_name,row_count
0,dim_clients,104
1,dim_content,519606
2,fact_daily,78835655
3,fact_daily_sample,11694072
4,fact_query_90d,2414248


In [17]:
schemas = {}

for name, source in TABLES.items():
    print(f"\n{'=' * 70}")
    print(f"TABLE: {name}")
    print(f"{'=' * 70}")

    schema = con.sql(f"""
        DESCRIBE
        SELECT *
        FROM {source}
    """).df()

    schemas[name] = schema
    display(schema)


TABLE: dim_clients


,column_name,column_type,null,key,default,extra
0,client_hash_id,VARCHAR,YES,None,None,None
1,is_active,BOOLEAN,YES,None,None,None
2,has_gsc_access,BOOLEAN,YES,None,None,None
3,has_ga4_access,BOOLEAN,YES,None,None,None
4,access_profile,VARCHAR,YES,None,None,None
5,client_created_date,DATE,YES,None,None,None
6,client_updated_date,DATE,YES,None,None,None
7,gsc_data_start,DATE,YES,None,None,None
8,ga4_data_start,DATE,YES,None,None,None



TABLE: dim_content


,column_name,column_type,null,key,default,extra
0,client_hash_id,VARCHAR,YES,None,None,None
1,content_hash_id,VARCHAR,YES,None,None,None
2,keyword_hash_id,VARCHAR,YES,None,None,None
3,url_hash_id,VARCHAR,YES,None,None,None
4,keyword_char_count,BIGINT,YES,None,None,None
5,keyword_token_count,BIGINT,YES,None,None,None
6,url_char_count,BIGINT,YES,None,None,None
7,content_created_date,DATE,YES,None,None,None
8,content_updated_date,DATE,YES,None,None,None
9,content_type,VARCHAR,YES,None,None,None



TABLE: fact_daily


,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None



TABLE: fact_daily_sample


,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None



TABLE: fact_query_90d


,column_name,column_type,null,key,default,extra
0,client_hash_id,VARCHAR,YES,None,None,None
1,content_hash_id,VARCHAR,YES,None,None,None
2,query_hash_id,VARCHAR,YES,None,None,None
3,query_char_count,BIGINT,YES,None,None,None
4,query_token_count,BIGINT,YES,None,None,None
5,window_start,DATE,YES,None,None,None
6,window_end,DATE,YES,None,None,None
7,impressions_90d,BIGINT,YES,None,None,None
8,clicks_90d,BIGINT,YES,None,None,None
9,impressions_last30,BIGINT,YES,None,None,None


In [18]:
samples = {}

for name, source in TABLES.items():
    print(f"\n{'=' * 70}")
    print(f"SAMPLE: {name}")
    print(f"{'=' * 70}")

    sample = con.sql(f"""
        SELECT *
        FROM {source}
        LIMIT 5
    """).df()

    samples[name] = sample
    display(sample)


SAMPLE: dim_clients


,client_hash_id,is_active,has_gsc_access,has_ga4_access,access_profile,client_created_date,client_updated_date,gsc_data_start,ga4_data_start
0,client_04660893ae39614a,True,True,True,gsc_and_ga4,2026-04-15,2026-06-27,NaT,2026-05-22
1,client_05475c07ed21a83a,True,False,False,no_search_or_analytics_access,2026-04-01,2026-06-27,NaT,NaT
2,client_06d356715a8ff3b6,True,True,True,gsc_and_ga4,2026-03-23,2026-07-05,2026-04-10,2026-04-06
3,client_0797ff3a1fc9a6a5,True,False,False,no_search_or_analytics_access,2025-05-26,2026-06-27,2025-11-05,NaT
4,client_08a6a72ff48e62c0,True,True,False,gsc_only,2025-05-26,2026-06-27,2025-09-24,NaT



SAMPLE: dim_content


,client_hash_id,content_hash_id,keyword_hash_id,url_hash_id,keyword_char_count,keyword_token_count,url_char_count,content_created_date,content_updated_date,content_type,search_volume,competition,competition_level,cpc,main_intent,backlinks,category_count,keyword_created_date,provider_used,model_used,char_count,word_count,last_optimized_date,optimization_eligible_date,is_published,is_deleted
0,client_04660893ae39614a,content_004de9653278b5a4,keyword_e754999ab88dd9f2,url_d6091f18cf628794,22,4,108,2026-05-30,2026-07-01,keyword article,30,0.91,HIGH,0.98,transactional,16,3,2026-05-12,gemini-generate-content,gemini-3-flash-preview,15682,2555,NaT,NaT,True,False
1,client_04660893ae39614a,content_00dc5efae381b2ab,keyword_4329d7aede8e208b,url_3a66d2f2e36823ca,31,6,95,2026-06-12,2026-07-01,keyword article,10,0.00,LOW,0.00,commercial,0,4,2026-06-01,gemini-generate-content,gemini-3-flash-preview,15438,2430,NaT,NaT,True,False
2,client_04660893ae39614a,content_01410f2556c327ac,keyword_9b08047d3d2a0406,url_809eda7a7e20b3b2,22,5,82,2026-05-09,2026-07-01,keyword article,480,0.36,MEDIUM,0.62,informational,169,4,2026-05-06,gemini-generate-content,gemini-3-flash-preview,16576,2645,NaT,NaT,True,False
3,client_04660893ae39614a,content_019f27f634053ca7,keyword_e7cec7ab1804c1c2,url_5fb42bafc4399861,14,3,92,2026-06-15,2026-06-15,keyword article,0,0.00,LOW,0.00,transactional,0,4,2026-06-01,gemini-generate-content,gemini-3-flash-preview,15457,2522,NaT,NaT,True,False
4,client_04660893ae39614a,content_01efa71faea45dcc,keyword_56b0062a1d8b7524,url_ece0abc3e5fb75f9,24,6,98,2026-05-21,2026-06-01,keyword article,2400,0.70,HIGH,0.90,transactional,52,4,2026-05-12,gemini-generate-content,gemini-3-flash-preview,15776,2552,NaT,NaT,True,False



SAMPLE: fact_daily


,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,gsc_avg_position,ga4_pageviews,ga4_sessions,ga4_users,ga4_engaged_sessions,ga4_total_engagement_sec,sessions_organic,sessions_direct,sessions_referral,sessions_social,sessions_paid,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2025-01-27,client_9958f0a7ae1df715,content_3b70a18ea133b2bb,True,True,True,False,30,0,115,3.833333,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,2025-01
1,2025-01-27,client_9958f0a7ae1df715,content_fe8e8155ce1d47a2,True,True,True,False,5,0,358,71.600000,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,2025-01
2,2025-01-27,client_9958f0a7ae1df715,content_b4462a1b90640058,True,True,True,False,1,0,34,34.000000,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,2025-01
3,2025-01-27,client_9958f0a7ae1df715,content_c899aef92518c714,True,True,True,False,6,0,140,23.333333,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,2025-01
4,2025-01-27,client_9958f0a7ae1df715,content_c7c1d2e68d9d0964,True,True,True,False,5,0,89,17.800000,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,2025-01



SAMPLE: fact_daily_sample


,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,gsc_avg_position,ga4_pageviews,ga4_sessions,ga4_users,ga4_engaged_sessions,ga4_total_engagement_sec,sessions_organic,sessions_direct,sessions_referral,sessions_social,sessions_paid,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-06-01,client_3ffa76342f366962,content_1a6296faee432dae,True,True,False,False,0,0,0,NaN,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,2026-06
1,2026-06-01,client_3ffa76342f366962,content_73f21e612565035a,True,True,False,False,0,0,0,NaN,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,2026-06
2,2026-06-01,client_3ffa76342f366962,content_5a5be514ff559598,True,True,False,False,0,0,0,NaN,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,2026-06
3,2026-06-01,client_3ffa76342f366962,content_05b377d0c8a5cfd8,True,True,False,False,0,0,0,NaN,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,2026-06
4,2026-06-01,client_3ffa76342f366962,content_dc34c661d63e55a9,True,True,False,False,0,0,0,NaN,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,2026-06



SAMPLE: fact_query_90d


,client_hash_id,content_hash_id,query_hash_id,query_char_count,query_token_count,window_start,window_end,impressions_90d,clicks_90d,impressions_last30,clicks_last30,impressions_prev30,clicks_prev30,avg_position_90d,avg_position_last30,avg_position_prev30,content_total_impressions_90d,content_visible_query_count,rare_query_count,rare_impressions_share,anonymized_impressions_share
0,client_08a6a72ff48e62c0,content_447894f2faf0d2bc,query_58b1b001f839d699,17,3,2026-04-02,2026-06-30,11,0,0,0,11,0,10.818182,NaN,10.818182,1466,14,32,0.043656,0.725102
1,client_08a6a72ff48e62c0,content_447894f2faf0d2bc,query_922b8eca2a24cd34,34,7,2026-04-02,2026-06-30,13,0,0,0,1,0,1.769231,NaN,11.000000,1466,14,32,0.043656,0.725102
2,client_08a6a72ff48e62c0,content_447894f2faf0d2bc,query_9f0c36a6ae2a6a99,16,2,2026-04-02,2026-06-30,16,0,11,0,5,0,23.562500,24.272727,22.000000,1466,14,32,0.043656,0.725102
3,client_08a6a72ff48e62c0,content_447894f2faf0d2bc,query_a032820b5467e996,24,4,2026-04-02,2026-06-30,55,0,1,0,1,0,2.200000,13.000000,0.000000,1466,14,32,0.043656,0.725102
4,client_08a6a72ff48e62c0,content_447894f2faf0d2bc,query_ba1a2f131961c5da,18,3,2026-04-02,2026-06-30,14,0,0,0,0,0,3.428571,NaN,NaN,1466,14,32,0.043656,0.725102


In [19]:
dim_clients_grain = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT client_hash_id) AS unique_clients,
        COUNT(*) - COUNT(DISTINCT client_hash_id) AS duplicate_client_rows
    FROM {TABLES["dim_clients"]}
""").df()

dim_clients_grain

,total_rows,unique_clients,duplicate_client_rows
0,104,104,0


In [20]:
dim_content_grain = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(
            DISTINCT client_hash_id || '|' || content_hash_id
        ) AS unique_client_content_pairs,
        COUNT(*) - COUNT(
            DISTINCT client_hash_id || '|' || content_hash_id
        ) AS duplicate_pairs
    FROM {TABLES["dim_content"]}
""").df()

dim_content_grain

,total_rows,unique_client_content_pairs,duplicate_pairs
0,519606,519606,0


In [21]:
fact_daily_grain = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(
            DISTINCT
            CAST(report_date AS VARCHAR)
            || '|'
            || client_hash_id
            || '|'
            || content_hash_id
        ) AS unique_daily_records,
        COUNT(*) - COUNT(
            DISTINCT
            CAST(report_date AS VARCHAR)
            || '|'
            || client_hash_id
            || '|'
            || content_hash_id
        ) AS duplicate_daily_records
    FROM {TABLES["fact_daily"]}
    WHERE month = '2026-03'
""").df()

fact_daily_grain

,total_rows,unique_daily_records,duplicate_daily_records
0,9841378,9841378,0


In [22]:
fact_query_grain = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(
            DISTINCT
            client_hash_id
            || '|'
            || content_hash_id
            || '|'
            || query_hash_id
            || '|'
            || CAST(window_start AS VARCHAR)
            || '|'
            || CAST(window_end AS VARCHAR)
        ) AS unique_query_window_records,
        COUNT(*) - COUNT(
            DISTINCT
            client_hash_id
            || '|'
            || content_hash_id
            || '|'
            || query_hash_id
            || '|'
            || CAST(window_start AS VARCHAR)
            || '|'
            || CAST(window_end AS VARCHAR)
        ) AS duplicate_query_records
    FROM {TABLES["fact_query_90d"]}
""").df()

fact_query_grain

,total_rows,unique_query_window_records,duplicate_query_records
0,2414248,2414248,0


In [23]:
daily_date_range = con.sql(f"""
    SELECT
        MIN(report_date) AS min_report_date,
        MAX(report_date) AS max_report_date,
        COUNT(DISTINCT month) AS month_count
    FROM {TABLES["fact_daily"]}
""").df()

daily_date_range

,min_report_date,max_report_date,month_count
0,2025-01-27,2026-06-30,18


In [24]:
query_windows = con.sql(f"""
    SELECT
        window_start,
        window_end,
        COUNT(*) AS row_count,
        COUNT(DISTINCT client_hash_id) AS clients,
        COUNT(DISTINCT content_hash_id) AS contents,
        COUNT(DISTINCT query_hash_id) AS queries
    FROM {TABLES["fact_query_90d"]}
    GROUP BY
        window_start,
        window_end
    ORDER BY window_end
""").df()

query_windows

,window_start,window_end,row_count,clients,contents,queries
0,2026-04-02,2026-06-30,2414248,52,133852,1180090


In [25]:
join_coverage = con.sql(f"""
    WITH daily_contents AS (
        SELECT DISTINCT
            client_hash_id,
            content_hash_id
        FROM {TABLES["fact_daily"]}
        WHERE report_date BETWEEN DATE '2026-04-02'
                              AND DATE '2026-06-30'
          AND gsc_data_available IS TRUE
    ),
    query_contents AS (
        SELECT DISTINCT
            client_hash_id,
            content_hash_id
        FROM {TABLES["fact_query_90d"]}
    )
    SELECT
        COUNT(*) AS daily_content_count,
        COUNT(q.content_hash_id) AS matched_content_count,
        COUNT(*) - COUNT(q.content_hash_id) AS unmatched_daily_contents,
        ROUND(
            100.0 * COUNT(q.content_hash_id) / COUNT(*),
            2
        ) AS match_rate_pct
    FROM daily_contents d
    LEFT JOIN query_contents q
        ON d.client_hash_id = q.client_hash_id
       AND d.content_hash_id = q.content_hash_id
""").df()

join_coverage

,daily_content_count,matched_content_count,unmatched_daily_contents,match_rate_pct
0,270645,133844,136801,49.45


In [26]:
daily_columns = con.sql(f"""
    DESCRIBE
    SELECT *
    FROM {TABLES["fact_daily"]}
""").df()

daily_columns

,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


In [6]:
content_columns = con.sql(f"""
    DESCRIBE
    SELECT *
    FROM {TABLES["dim_content"]}
""").df()

content_columns

,column_name,column_type,null,key,default,extra
0,client_hash_id,VARCHAR,YES,None,None,None
1,content_hash_id,VARCHAR,YES,None,None,None
2,keyword_hash_id,VARCHAR,YES,None,None,None
3,url_hash_id,VARCHAR,YES,None,None,None
4,keyword_char_count,BIGINT,YES,None,None,None
5,keyword_token_count,BIGINT,YES,None,None,None
6,url_char_count,BIGINT,YES,None,None,None
7,content_created_date,DATE,YES,None,None,None
8,content_updated_date,DATE,YES,None,None,None
9,content_type,VARCHAR,YES,None,None,None


In [7]:
label_check = con.sql(f"""
WITH content_windows AS (
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(gsc_impressions) FILTER (
            WHERE report_date BETWEEN DATE '2026-03-02'
                                  AND DATE '2026-03-31'
        ) AS impressions_last30,

        SUM(gsc_impressions) FILTER (
            WHERE report_date BETWEEN DATE '2026-04-01'
                                  AND DATE '2026-04-30'
        ) AS impressions_future30

    FROM {TABLES["fact_daily"]}

    WHERE report_date BETWEEN DATE '2026-03-02'
                          AND DATE '2026-04-30'
      AND gsc_data_available IS TRUE

    GROUP BY
        client_hash_id,
        content_hash_id
)

SELECT
    *,
    (impressions_future30 - impressions_last30) * 1.0
        / NULLIF(impressions_last30, 0) AS future_change_pct

FROM content_windows

WHERE impressions_last30 > 0
  AND impressions_future30 IS NOT NULL
""").df()

label_check.head()

,client_hash_id,content_hash_id,impressions_last30,impressions_future30,future_change_pct
0,client_73cda7b4e4f265ea,content_1c5b788f459efc0f,362.0,325.0,-0.102210
1,client_73cda7b4e4f265ea,content_f2247734067613ef,278.0,177.0,-0.363309
2,client_73cda7b4e4f265ea,content_c892bc4727c915fd,29.0,6.0,-0.793103
3,client_73cda7b4e4f265ea,content_d1e5d0a7df416b6b,221.0,243.0,0.099548
4,client_73cda7b4e4f265ea,content_98c2734f085edb92,177.0,82.0,-0.536723
